In [ ]:
# Import library dasar
import os
import random
import shutil
import zipfile
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Set random seed agar hasil balancing dan splitting konsisten
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Initial setup completed. Seed has been fixed.")

Initial setup completed. Seed has been fixed.


In [ ]:
# Install gdown untuk download file dari Google Drive
!pip install gdown -q

import gdown

# File ID dari Google Drive
file_id = "1KY9QleBSGq31An1hnkQ7k2h4-NeFG5rK"
url = f"https://drive.google.com/uc?id={file_id}"

# Nama file zip output
output_zip = "original_dataset.zip"

# Download file
gdown.download(url, output_zip, quiet=False)

print("Dataset downloaded successfully.")

Downloading...
From (original): https://drive.google.com/uc?id=1KY9QleBSGq31An1hnkQ7k2h4-NeFG5rK
From (redirected): https://drive.google.com/uc?id=1KY9QleBSGq31An1hnkQ7k2h4-NeFG5rK&confirm=t&uuid=2a975984-ef6f-4937-ba08-3cceab2cc1d5
To: /content/original_dataset.zip
100%|██████████| 499M/499M [00:03<00:00, 131MB/s]

Dataset downloaded successfully.


In [ ]:
# Path folder dataset mentah
RAW_DIR = "/content/raw_dataset"

# Hapus folder lama jika ada
if os.path.exists(RAW_DIR):
    shutil.rmtree(RAW_DIR)

os.makedirs(RAW_DIR, exist_ok=True)

# Extract zip
with zipfile.ZipFile(output_zip, "r") as zip_ref:
    zip_ref.extractall(RAW_DIR)

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [ ]:
# Cek struktur folder dataset mentah
for root, dirs, files in os.walk(RAW_DIR):
    level = root.replace(RAW_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    if level < 3:
        for f in files[:5]:
            print(f"{indent}  {f}")

raw_dataset/
  AiArtData/
    AiArtData/
      filippo-nassetti-ai-designs-spaceships-bosch-caravaggio-designboom-600.jpg
      GettyImages-1096829784-copy.jpg
      image33.jpg
      Blogs.jpg
      tsia-navigating-ai-landscape-blog-hero.png
  RealArt/
    RealArt/
      pexels-mali-maeder-110820-1280x720.png
      HD-wallpaper-scenery-lake-nature-sky-tree-water.jpg
      Portrait-Photographers-Yousuf-Karsh-King.jpg
      7ce6zbax23_WEB_257649.jpg
      set-safari-animals-illustration-600nw-2150029883.jpg


In [ ]:
# Folder output untuk dataset yang sudah dirapikan
FIX_DIR = "/content/dataset_fix"

# Hapus folder lama jika ada
if os.path.exists(FIX_DIR):
    shutil.rmtree(FIX_DIR)

os.makedirs(os.path.join(FIX_DIR, "AI"), exist_ok=True)
os.makedirs(os.path.join(FIX_DIR, "Real"), exist_ok=True)

# Ekstensi gambar yang diterima
valid_extensions = (".jpg", ".jpeg", ".png", ".webp")

# Counter untuk menghindari nama file bentrok
image_counter = {
    "AI": 0,
    "Real": 0
}

# Fungsi deteksi class dari path folder
def detect_class_from_path(path):
    path_lower = path.lower()

    # Keyword untuk class AI
    ai_keywords = ["ai", "fake", "generated", "synthetic"]

    # Keyword untuk class Real
    real_keywords = ["real", "human", "natural", "original"]

    if any(keyword in path_lower for keyword in ai_keywords):
        return "AI"
    elif any(keyword in path_lower for keyword in real_keywords):
        return "Real"
    else:
        return None

# Scan semua gambar dari dataset mentah
for root, dirs, files in os.walk(RAW_DIR):
    files = sorted(files)

    detected_class = detect_class_from_path(root)

    for file in files:
        if file.lower().endswith(valid_extensions) and detected_class is not None:
            src_path = os.path.join(root, file)

            # Nama file tetap pakai nama asli, hanya ditambah prefix angka jika bentrok
            file_base, file_ext = os.path.splitext(file)
            new_file_name = file

            dst_path = os.path.join(FIX_DIR, detected_class, new_file_name)

            # Kalau nama file sudah ada, tambahkan nomor supaya tidak overwrite
            if os.path.exists(dst_path):
                new_file_name = f"{image_counter[detected_class]}_{file}"
                dst_path = os.path.join(FIX_DIR, detected_class, new_file_name)

            try:
                img = Image.open(src_path).convert("RGB")
                img.save(dst_path)
                image_counter[detected_class] += 1
            except Exception as e:
                print(f"Skipped file: {src_path}")

print("Dataset folders fixed successfully.")
print("AI images:", image_counter["AI"])
print("Real images:", image_counter["Real"])

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Dataset folders fixed successfully.
AI images: 539
Real images: 434


In [ ]:
# Cek jumlah gambar setelah folder dirapikan
for class_name in ["AI", "Real"]:
    class_path = os.path.join(FIX_DIR, class_name)

    total_images = len([
        f for f in os.listdir(class_path)
        if f.lower().endswith(valid_extensions)
    ])

    print(f"{class_name}: {total_images} images")

AI: 539 images
Real: 434 images


In [ ]:
# Folder output untuk dataset balanced
BALANCED_DIR = "/content/dataset_balanced"

# Hapus folder lama jika ada
if os.path.exists(BALANCED_DIR):
    shutil.rmtree(BALANCED_DIR)

os.makedirs(BALANCED_DIR, exist_ok=True)

# Nama class
class_names = ["AI", "Real"]

# Ambil file per class dengan sorted agar urutannya konsisten
files_per_class = {}

for class_name in class_names:
    class_path = os.path.join(FIX_DIR, class_name)

    files = sorted([
        f for f in os.listdir(class_path)
        if f.lower().endswith(valid_extensions)
    ])

    files_per_class[class_name] = files

# Cari jumlah gambar paling sedikit
min_count = min(len(files_per_class[class_name]) for class_name in class_names)

print(f"Minimum class count: {min_count}")

# Random generator dengan seed tetap
rng = random.Random(SEED)

# Sampling jumlah yang sama untuk setiap class
for class_name in class_names:
    source_class_dir = os.path.join(FIX_DIR, class_name)
    target_class_dir = os.path.join(BALANCED_DIR, class_name)

    os.makedirs(target_class_dir, exist_ok=True)

    selected_files = rng.sample(files_per_class[class_name], min_count)
    selected_files = sorted(selected_files)

    for file in selected_files:
        src_path = os.path.join(source_class_dir, file)
        dst_path = os.path.join(target_class_dir, file)
        shutil.copy(src_path, dst_path)

print("Dataset balanced successfully.")

for class_name in class_names:
    total = len([
        f for f in os.listdir(os.path.join(BALANCED_DIR, class_name))
        if f.lower().endswith(valid_extensions)
    ])
    print(f"{class_name}: {total} images")

Minimum class count: 434
Dataset balanced successfully.
AI: 434 images
Real: 434 images


In [ ]:
# Folder output final
OUTPUT_DIR = "/content/dataset_split_224"

# Pengaturan resize dan split
IMAGE_SIZE = (224, 224)

# Hapus folder final lama jika ada
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

# Membuat struktur folder final
for split in ["train", "val", "test"]:
    for class_name in class_names:
        os.makedirs(os.path.join(OUTPUT_DIR, split, class_name), exist_ok=True)

# Split dan resize per class
for class_name in class_names:
    class_path = os.path.join(BALANCED_DIR, class_name)

    images = sorted([
        f for f in os.listdir(class_path)
        if f.lower().endswith(valid_extensions)
    ])

    # Split 70% train dan 30% sementara
    train_files, temp_files = train_test_split(
        images,
        test_size=0.30,
        random_state=SEED,
        shuffle=True
    )

    # Split 30% sementara menjadi 15% val dan 15% test
    val_files, test_files = train_test_split(
        temp_files,
        test_size=0.50,
        random_state=SEED,
        shuffle=True
    )

    split_data = {
        "train": sorted(train_files),
        "val": sorted(val_files),
        "test": sorted(test_files)
    }

    for split_name, file_list in split_data.items():
        output_class_dir = os.path.join(OUTPUT_DIR, split_name, class_name)

        for file_name in tqdm(file_list, desc=f"{class_name} - {split_name}"):
            src_path = os.path.join(class_path, file_name)
            dst_path = os.path.join(output_class_dir, file_name)

            try:
                img = Image.open(src_path).convert("RGB")
                img = img.resize(IMAGE_SIZE)
                img.save(dst_path)
            except Exception as e:
                print(f"Skipped file: {src_path}")

print("Dataset split and resize completed successfully.")

Real - test: 100%|██████████| 66/66 [00:01<00:00, 43.29it/s]

Dataset split and resize completed successfully.


In [ ]:
# Cek jumlah gambar final di train, val, test
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()} SET")

    for class_name in class_names:
        class_path = os.path.join(OUTPUT_DIR, split, class_name)

        total_images = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith(valid_extensions)
        ])

        print(f"{class_name}: {total_images} images")


TRAIN SET
AI: 303 images
Real: 303 images

VAL SET
AI: 65 images
Real: 65 images

TEST SET
AI: 66 images
Real: 66 images


In [ ]:
# Cek ukuran beberapa gambar hasil resize
for split in ["train", "val", "test"]:
    for class_name in class_names:
        class_path = os.path.join(OUTPUT_DIR, split, class_name)
        files = sorted([
            f for f in os.listdir(class_path)
            if f.lower().endswith(valid_extensions)
        ])

        if len(files) > 0:
            sample_path = os.path.join(class_path, files[0])
            img = Image.open(sample_path)
            print(f"{split}/{class_name}: {img.size}")

print("Image size verification completed.")

train/AI: (224, 224)
train/Real: (224, 224)
val/AI: (224, 224)
val/Real: (224, 224)
test/AI: (224, 224)
test/Real: (224, 224)
Image size verification completed.


In [ ]:
# Compress dataset final menjadi zip
ZIP_OUTPUT_NAME = "/content/dataset_split_224"

shutil.make_archive(ZIP_OUTPUT_NAME, "zip", OUTPUT_DIR)

print("Dataset zip created successfully.")
print("File name: dataset_split_224.zip")

Dataset zip created successfully.
File name: dataset_split_224.zip


In [ ]:
# Download file zip dataset final
from google.colab import files

files.download("/content/dataset_split_224.zip")

print("Dataset zip is ready to download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dataset zip is ready to download.
